In [27]:
#basic libraries
import pandas as pd
import numpy as np
import matplotlib as plt

#preprocessing libraries
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

#ML models
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

#ANN libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


In [28]:
#import data
df = pd.read_csv('credit_cart_data_preprocessed.csv')
df.head()

,Unnamed: 0,Credit Amount,gender,Education level,Marital Status,Age,Y,totall bill statement,total amount paid,avg repayment
0,1,20000,2,2,1,24,1,7704,689,-0.0
1,2,120000,2,2,2,26,1,17077,5000,0.0
2,3,90000,2,2,2,34,0,101653,11018,0.0
3,4,50000,2,2,1,37,0,231334,8388,0.0
4,5,50000,1,2,1,57,0,109339,59049,-0.0


In [29]:
#drop first column
df.drop(['Unnamed: 0'], axis =1, inplace= True)

In [30]:
#split in X and Y
X = df.drop(['Y'], axis= 1)
y = df['Y']

In [31]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42)

In [32]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 23972 entries, 27330 to 23654
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Credit Amount          23972 non-null  int64  
 1   gender                 23972 non-null  int64  
 2   Education level        23972 non-null  int64  
 3   Marital Status         23972 non-null  int64  
 4   Age                    23972 non-null  int64  
 5   totall bill statement  23972 non-null  int64  
 6   total amount paid      23972 non-null  int64  
 7   avg repayment          23972 non-null  float64
dtypes: float64(1), int64(7)
memory usage: 1.6 MB


In [34]:
#columns for standardization
train_df_num_features = X_train[['Credit Amount', 'Age', 'totall bill statement','total amount paid', 'avg repayment']]
test_df_num_features = X_test[['Credit Amount', 'Age', 'totall bill statement','total amount paid', 'avg repayment']]


In [35]:
#scalling

preprocessor = StandardScaler()

X_train_scaled = preprocessor.fit_transform(train_df_num_features)
X_test_scaled = preprocessor.transform(test_df_num_features)


In [36]:
#adding scalled features in removing non scaled features
X_train.drop(['Credit Amount', 'Age', 'totall bill statement','total amount paid', 'avg repayment'], axis= 1, inplace= True)
X_test.drop(['Credit Amount', 'Age', 'totall bill statement','total amount paid', 'avg repayment'], axis= 1, inplace= True)

In [ ]:
#adding scalled features


In [20]:
#save scalller as pickle
import joblib
joblib.dump(preprocessor, 'scaler.pkl')

['scaler.pkl']

In [8]:
#train ML models

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def model_evaluator (true, predicted):
    ac = accuracy_score(true, predicted)
    cf = confusion_matrix(true, predicted)
    report = classification_report(true, predicted)

    return ac, cf, report

In [9]:
models = {'Logistic Regression': LogisticRegression(),
          'Random Forest': RandomForestClassifier(),
          'XGB' : XGBClassifier ()}

In [10]:
#iterate and predict
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    #evaluate models

    X_train_accuracy, X_train_CF, X_train_Report = model_evaluator(y_train, y_pred_train)
    X_test_accuracy, X_test_CF, X_test_Report = model_evaluator(y_test, y_pred_test)

    print(list(models.keys())[i])

    print('-'*20)
    print('for train data')
    print('-'*20)

    print(f'accuracy_score of the model is: ', X_train_accuracy)
    print(f'confusion matrix of  the model is\n:', X_train_CF)
    print(f'classificaiton report of the model is:\n', X_train_Report)

    print('-'*20)
    print('for test data')
    print('-'*20)
    
    print(f'accuracy_score of the model is: ', X_test_accuracy)
    print(f'confusion matrix of  the model is\n:', X_test_CF)
    print(f'classificaiton report of the model is:\n', X_test_Report)


    print('='*30)

Logistic Regression
--------------------
for train data
--------------------
accuracy_score of the model is:  0.7965125980310362
confusion matrix of  the model is
: [[18308   354]
 [ 4524   786]]
classificaiton report of the model is:
               precision    recall  f1-score   support

           0       0.80      0.98      0.88     18662
           1       0.69      0.15      0.24      5310

    accuracy                           0.80     23972
   macro avg       0.75      0.56      0.56     23972
weighted avg       0.78      0.80      0.74     23972

--------------------
for test data
--------------------
accuracy_score of the model is:  0.7932588019355915
confusion matrix of  the model is
: [[4582   91]
 [1148  172]]
classificaiton report of the model is:
               precision    recall  f1-score   support

           0       0.80      0.98      0.88      4673
           1       0.65      0.13      0.22      1320

    accuracy                           0.79      5993
   macro

ANN modeling

In [11]:
ANN_model = Sequential([
    Dense(64, activation= 'relu', input_shape = (X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation= 'sigmoid')
])

ANN_model.summary()

c:\Users\Shubham\Desktop\Data Science Prep\DS Projects\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,689 (10.50 KB)

 Trainable params: 2,689 (10.50 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
#compiling models with optimier and loss function
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import binary_crossentropy

optimizer = Adam(learning_rate= 0.01)
loss_fun = binary_crossentropy

ANN_model.compile(optimizer= optimizer, loss= binary_crossentropy, metrics= ['accuracy'])


In [15]:
#setup early stopping
early_stopping_calllback = EarlyStopping(monitor= 'val_loss', patience= 10, restore_best_weights= 'True')

In [ ]:
#training model
history = ANN_model.fit(X_train, y_train,
    validation_data = (X_test, y_test),
    epochs = 100,
    callbacks = [early_stopping_calllback]
)

Epoch 1/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.7996 - loss: 0.4778 - val_accuracy: 0.7939 - val_loss: 0.4750
Epoch 2/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8001 - loss: 0.4672 - val_accuracy: 0.8039 - val_loss: 0.4631
Epoch 3/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8017 - loss: 0.4644 - val_accuracy: 0.7968 - val_loss: 0.4619
Epoch 4/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8023 - loss: 0.4623 - val_accuracy: 0.7966 - val_loss: 0.4637
Epoch 5/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8030 - loss: 0.4620 - val_accuracy: 0.7983 - val_loss: 0.4628
Epoch 6/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8024 - loss: 0.4617 - val_accuracy: 0.7998 - val_loss: 0.4606
Epoch 7/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8025 - loss: 0.4596 - val_accuracy: 0.7959 - val_loss: 0.4612
Epoch 8/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8024 - loss: 0.4589 - val_accu

In [19]:
#saving model
ANN_model.save('model.h5')